In [1]:
import pandas as pd


print(pd.read_csv('results_all.csv'))

      Model  Accuracy      F1  ROC-AUC  Precision           Saved_Time
0  CatBoost    0.8294  0.7817   0.8830     0.7671  2026-06-04 13:00:18
1   XGBoost    0.8395  0.7935   0.8870     0.7834  2026-06-04 14:53:42
2       MLP    0.8159  0.7310   0.8678     0.8344  2026-06-10 15:22:05


In [2]:
from core import fit_preprocessor, transform_preprocessor

import pandas as pd
import pickle


df_train_raw = pd.read_csv('train.csv')

X_train, Y_train, preprocess_params = fit_preprocessor(df_train_raw)

import pickle
with open('preprocess_params.pkl', 'wb') as f:
    pickle.dump(preprocess_params, f)

In [3]:
X_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Pclass        891 non-null    int64  
 1   Sex           891 non-null    int64  
 2   Age           891 non-null    float64
 3   Fare          891 non-null    float64
 4   Cab_count     891 non-null    int64  
 5   Emb_Q         891 non-null    int64  
 6   Emb_S         891 non-null    int64  
 7   Deck_tar_enc  891 non-null    float64
 8   Designation   891 non-null    int64  
 9   Family        891 non-null    int64  
 10  Ticket_uniq   891 non-null    int64  
dtypes: float64(3), int64(8)
memory usage: 76.7 KB


In [4]:
with open('preprocess_params.pkl', 'rb') as f:
    preprocess_params = pickle.load(f)

df_test_raw = pd.read_csv('test.csv')
passenger_ids = df_test_raw['PassengerId']
X_test = transform_preprocessor(df_test_raw, preprocess_params)

true_submission = pd.read_csv('gender_submission.csv')
y_true = true_submission['Survived']

In [5]:
X_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Pclass        418 non-null    int64  
 1   Sex           418 non-null    int64  
 2   Age           418 non-null    float64
 3   Fare          418 non-null    float64
 4   Cab_count     418 non-null    int64  
 5   Emb_Q         418 non-null    int64  
 6   Emb_S         418 non-null    int64  
 7   Deck_tar_enc  418 non-null    float64
 8   Designation   418 non-null    int64  
 9   Family        418 non-null    int64  
 10  Ticket_uniq   418 non-null    int64  
dtypes: float64(3), int64(8)
memory usage: 36.1 KB


In [6]:
import joblib
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix


def evaluate_models_on_test(model_paths, X_test, y_true, passenger_ids):
    """Загрузка моделей -> предсказания -> метрики"""

    results = []
    predictions = {}

    for name, path in model_paths.items():
        print(f'Загрузка модели {name} из {path}...')
        model = joblib.load(path)

        print(f'Предсказание для {name}...')
        preds = model.predict(X_test)
        predictions[name] = preds

        acc = accuracy_score(y_true, preds)
        f1 = f1_score(y_true, preds)
        prec = precision_score(y_true, preds)
        rec = recall_score(y_true, preds)
        cm = confusion_matrix(y_true, preds)

        tn, fp, fn, tp = cm.ravel()

        results.append({
            'Model': name,
            'Accuracy': acc,
            'F1': f1,
            'Precision': prec,
            'Recall': rec,
            'TN': tn,
            'FP': fp,
            'FN': fn,
            'TP': tp
        })

        print(f'  Accuracy: {acc:.4f}, F1: {f1:.4f}')
        print(f'  Confusion matrix:\n{cm}')
        print('-' * 50)
    
    results_df = pd.DataFrame(results)
    results_df = results_df.round(4)

    return results_df, predictions


In [7]:
model_paths = {
    'CatBoost': 'models/CatBoost_best.pkl',
    'XGBoost': 'models/XGBoost_best.pkl',
    'MLP': 'models/MLP_best.pkl'
}

results_df, all_preds = evaluate_models_on_test(model_paths, X_test, y_true, passenger_ids)

print('\nСводная таблица метрик:')
print(results_df.to_string(index=False))

results_df.to_csv('test_models_comparison.csv', index=False)

for name, preds in all_preds.items():
    submission = pd.DataFrame({'PassengerId': passenger_ids, 'Survived': preds})
    submission.to_csv(f'submission_{name}.csv', index=False)

Загрузка модели CatBoost из models/CatBoost_best.pkl...
Предсказание для CatBoost...
  Accuracy: 0.8971, F1: 0.8709
  Confusion matrix:
[[230  36]
 [  7 145]]
--------------------------------------------------
Загрузка модели XGBoost из models/XGBoost_best.pkl...
Предсказание для XGBoost...
  Accuracy: 0.8732, F1: 0.8285
  Confusion matrix:
[[237  29]
 [ 24 128]]
--------------------------------------------------
Загрузка модели MLP из models/MLP_best.pkl...
Предсказание для MLP...
  Accuracy: 0.8971, F1: 0.8502
  Confusion matrix:
[[253  13]
 [ 30 122]]
--------------------------------------------------

Сводная таблица метрик:
   Model  Accuracy     F1  Precision  Recall  TN  FP  FN  TP
CatBoost    0.8971 0.8709     0.8011  0.9539 230  36   7 145
 XGBoost    0.8732 0.8285     0.8153  0.8421 237  29  24 128
     MLP    0.8971 0.8502     0.9037  0.8026 253  13  30 122


In [ ]:
true_submission = pd.read_csv('gender_submission.csv')

assert (test_submission['PassengerId'] == true_submission['PassengerId']).all(), 'ID не совпадают'

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

y_true = true_submission['Survived']
y_pred = test_submission['Survived']

acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f'Accuracy: {acc:.4f}')
print(f'F1-score: {f1:.4f}')
print('Confusion matrix:')
print(confusion_matrix(y_true, y_pred))

Accuracy: 0.8971
F1-score: 0.8709
Confusion matrix:
[[230  36]
 [  7 145]]
